# Import packages

In [ ]:
import joblib
import numpy
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ***__Built regression models__***

## Load the csv

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")

## Creating month and year datas

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

## Creating data preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["month"]),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            ["gare_depart", "gare_arrivee"],
        ),
    ]
)

We selected gare_depart, gare_arrivee, and month as the most relevant features based on their usefulness for prediction and dashboard compatibility.

## Labelization of the different datas

In [ ]:
x = df[["gare_depart", "gare_arrivee", "month"]]

y = numpy.array(df["retard_moyen_tous_trains_arrivee"], dtype=numpy.float64)

## Splitting datas to divide between test values and train values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=43
)

## Storing the efficiency and the pipelines into a dictionnary

In [ ]:
r2_results = {}
pipelines = {}

# ***Linear regression model***

## Creating the pipeline

In [ ]:
pipe_lr = Pipeline([("preprocessor", preprocessor), ("model", LinearRegression())])

## Training the model

In [ ]:
pipe_lr.fit(X_train, y_train)

## Prediction of the model

In [ ]:
pred_lr = pipe_lr.predict(X_test)

## Efficiency of the model

In [ ]:
rmse_score_lr = numpy.sqrt(mean_squared_error(y_test, pred_lr))
mae_lr = mean_absolute_error(y_test, pred_lr)
r2_score_lr = r2_score(y_test, pred_lr)

print("Linear Regression:")
print("RMSE :", rmse_score_lr)
print("MAE  :", mae_lr)
print("R²   :", r2_score_lr)

r2_results["Linear Regression"] = r2_score_lr
pipelines["Linear Regression"] = pipe_lr

## Adding hyperparameters to the model

In [ ]:
param_grid_lr = {
    "model__fit_intercept": [True, False],
}

grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring="r2", n_jobs=-1)

grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
pred_best_lr = best_lr.predict(X_test)

rmse_best_lr = numpy.sqrt(mean_squared_error(y_test, pred_best_lr))
mae_best_lr = mean_absolute_error(y_test, pred_best_lr)
r2_best_lr = r2_score(y_test, pred_best_lr)

print("Tuned Linear Regression:")
print("Best params:", grid_lr.best_params_)
print("RMSE:", rmse_best_lr)
print("MAE:", mae_best_lr)
print("R²:", r2_best_lr)

r2_results["Tuned Linear Regression"] = r2_best_lr
pipelines["Tuned Linear Regression"] = best_lr

# ***Random forest model***

## Creating the pipeline

In [ ]:
pipe_rf = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=43, n_jobs=-1)),
    ]
)
# random_state: a random number, n_jobs: use all the cores of the computer

## Training the model

In [ ]:
pipe_rf.fit(X_train, y_train)

## Prediction of the model

In [ ]:
pred_rf = pipe_rf.predict(X_test)

## Efficiency of the model

In [ ]:
rmse_score_rf = numpy.sqrt(mean_squared_error(y_test, pred_rf))
mae_rf = mean_absolute_error(y_test, pred_rf)
r2_score_rf = r2_score(y_test, pred_rf)

print("Random Forest:")
print("RMSE :", rmse_score_rf)
print("MAE  :", mae_rf)
print("R²   :", r2_score_rf)

r2_results["Random Forest"] = r2_score_rf
pipelines["Random Forest"] = pipe_rf

## Adding hyperparameters to the model

In [ ]:
param_grid_rf = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_split": [2, 5, 10],
}

grid_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring="r2", n_jobs=-1)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
pred_best_rf = best_rf.predict(X_test)

rmse_best_rf = numpy.sqrt(mean_squared_error(y_test, pred_best_rf))
mae_best_rf = mean_absolute_error(y_test, pred_best_rf)
r2_best_rf = r2_score(y_test, pred_best_rf)

print("Tuned Random Forest:")
print("Best params:", grid_rf.best_params_)
print("RMSE:", rmse_best_rf)
print("MAE:", mae_best_rf)
print("R²:", r2_best_rf)

r2_results["Tuned Random Forest"] = r2_best_rf
pipelines["Tuned Random Forest"] = best_rf

# ***Gradient boosting regressor***

## Creating the pipeline

In [ ]:
pipe_gbr = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(learning_rate=0.1, random_state=43)),
    ]
)
# learning rate: how much the program ajust its parameters, random_state: also a random number

## Training the model

In [ ]:
pipe_gbr.fit(X_train, y_train)

## Prediction of the model

In [ ]:
pred_gbr = pipe_gbr.predict(X_test)

## Efficiency of the model

In [ ]:
rmse_score_gbr = numpy.sqrt(mean_squared_error(y_test, pred_gbr))
mae_gbr = mean_absolute_error(y_test, pred_gbr)
r2_score_gbr = r2_score(y_test, pred_gbr)

print("Gradient boosting regressor:")
print("RMSE :", rmse_score_gbr)
print("MAE  :", mae_gbr)
print("R²   :", r2_score_gbr)

r2_results["Gradient Boosting Regressor"] = r2_score_gbr
pipelines["Gradient Boosting Regressor"] = pipe_gbr

## Adding hyperparameters to the model

In [ ]:
param_grid_gbr = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 8, None],
    "model__min_samples_split": [2, 5, 10],
    "model__learning_rate": [0.05, 0.1, 0.2],
}

grid_gbr = GridSearchCV(pipe_gbr, param_grid_gbr, cv=5, scoring="r2", n_jobs=-1)

grid_gbr.fit(X_train, y_train)

best_gbr = grid_gbr.best_estimator_
pred_best_gbr = best_gbr.predict(X_test)

rmse_best_gbr = numpy.sqrt(mean_squared_error(y_test, pred_best_gbr))
mae_best_gbr = mean_absolute_error(y_test, pred_best_gbr)
r2_best_gbr = r2_score(y_test, pred_best_gbr)

print("Tuned Gradient Boosting Regressor:")
print("Best params:", grid_gbr.best_params_)
print("RMSE:", rmse_best_gbr)
print("MAE:", mae_best_gbr)
print("R²:", r2_best_gbr)

r2_results["Tuned Gradient Boosting Regressor"] = r2_best_gbr
pipelines["Tuned Gradient Boosting Regressor"] = best_gbr

# Choosing the best model

In [ ]:
choosen_model = max(r2_results, key=r2_results.get)
print("Choosen model:", choosen_model)

The final model is selected based on the best R2 score.
It is also compatible with the dashboard because it only uses gare_depart,
gare_arrivee and month as input features.

## Save the model into a .pkl file

In [ ]:
joblib.dump(pipelines[choosen_model], "model.pkl")